## Explore all Fluree v4 branch management endpoints
### Create a brabch for a given ledger
```python
POST /branch
POST http://localhost:8090/v1/fluree/branch
```



In [1]:
import requests
url = "http://localhost:8090/v1/fluree/branch"
payload= {
    "ledger":"demo",
    "branch":"feature-test",
    "source": "dev"
}

response = requests.post(url, json=payload)
print(response.json())

{'error': 'Internal error: Source branch demo:dev has no commit head', 'status': 500, '@type': 'err:system/InternalError'}


**Note**: It is normal to face the error:
```json

{'error': 'Internal error: Source branch demo:dev has no commit head', 'status': 500, '@type': 'err:system/InternalError'}
```
- the source branch that is demo:dev has no latest commit pointing to any data. The branch demo:dev is there, but no commit was made.
- The commit history may have been removed accidentally.
- The application is looking for demo:dev, but the actual branch may be main, master, or another name.
- In distributed databases (such as Fluree), metadata references a branch that has not been fully synchronized.

### For a given ledger, list all the branches: 
```python
GET /branch/{ledger-name}
```


In [2]:
url = f"http://localhost:8090/v1/fluree/branch/demo"
response = requests.get(url)
print(response.json())
print(response.status_code)

[{'branch': 'dev', 'ledger_id': 'demo:dev', 't': 0}]
200


- branch: means branch name
- ledger_id: Full ledger:branch identifier
- t: Current transaction time on this branch
- source: Source branch (only present for branches created via /branch)

### Drop a branch
```python
POST /drop-branch
```
Drop a branch from a ledger. Admin-protected


In [4]:
url = f"http://localhost:8090/v1/fluree/drop-branch"
payload = {
    "ledger":"demo",
    "branch":"dev"
}

response = requests.post(url, json=payload)
print(response.json())
print(response.status_code)

{'error': "Cannot drop 'dev': it is the root of ledger 'demo'. Use drop_ledger to remove the whole ledger.", 'status': 400, '@type': 'err:system/InternalError'}
400


- ledger_id:  Full ledger:branch identifier of the dropped branch
- status: Drop status
- deferred: True if the branch has children: retracted but storage preserved
- filed_deleted: Number of storage artifacts removed; omitted when zero
- cascade: List of ancestors branch ledger_ids that were cascade-dropped; omitted when empty
- warnings: Any non-fatal warnings during the drop; omitted when empty

**Behavior**:

- Cannot drop main: Returns 400 bad request
- Leaf branch (no children): Fully drops, deletes storage artifacts, purges, Nsrecord, decrements parent's child count. If the parent was previously retracted and its child count reaches 0, the parent is cascade-dropped too
- Branch with children (branches > 0): Retracted (hidden from listings, rejects new transactions) but storage is preserved for children. When the last child is eventually dropped, the retracted parent is cascade-purged automatically.


## Drop a single named graph:
```python
POST /drop-graph
```

Drop a single named graph from one branch of a ledger by transactionally retracting every triple currently asserted under that graph IRI. The drop runs as normal commit at t=current_t+1: hostory at earlier t values is preserved. **Admin protected**



In [6]:
url = f"http://localhost:8090/v1/fluree/drop-graph"
payload = {
    "ledger":"demo:dev",
    "graph":"http://schema.org/"
}

response = requests.post(url, json=payload)
print(response.json())


{'error': "Not found: Named graph 'http://schema.org/' is not registered in ledger 'demo:dev'", 'status': 404, '@type': 'err:db/LedgerNotFound'}


**Behavior**:
- Transactional and history-preserving: A query as-of an earlier t still sees the graph populated.
- Per-branch: Only affects the targeted branch--sibling branches that share the same graph IRI are not touched.
- Refuses system graphs: The default graph, urn:fluree:{ledger_id}#txn-meta, and urn:fluree:{ledger_id}#config cannot be dropped (400 Bad Request).
- Refuses unknown graphs: Returns 404 when graph is not registered in the ledger's graph registry, the call never auto-registers a new graph slot.
- Idempotent: A second call on an already-empty graph returns **commited:false**, **retracted:0** without producing commit.

### Rebase a branch
Rebase a branch onto its source branch's current head. Admin protected
```json
POST /rebase
```


In [10]:
url = f"http://localhost:8090/v1/fluree/rebase"

payload:{
    "ledger": "demo",
    "branch": "dev",
    "strategy": "take-both"
}

response = requests.post(url, json=payload)
print(response.json())


{'error': 'Bad request: Invalid JSON: missing field `branch` at line 1 column 53', 'status': 400, '@type': 'err:api/BadRequest'}


## Merge branches
```json
POST /merge
```

Merge a source branch into a target branch. Admin protected


In [11]:
url = f"http://localhost:8090/v1/fluree/merge"
payload = {
  "ledger": "demo",
  "source": "feature-x",
  "target": "dev",
  "strategy": "take-both"
}

print(response.json())
print(response.status_code)

{'error': 'Bad request: Invalid JSON: missing field `branch` at line 1 column 53', 'status': 400, '@type': 'err:api/BadRequest'}
400


**Examples**:

```json
# Merge feature-x into its parent (inferred from branch point)
curl -X POST http://localhost:8090/v1/fluree/merge \
  -H "Content-Type: application/json" \
  -d '{"ledger": "mydb", "source": "feature-x"}'

# Merge dev into main (explicit target)
curl -X POST http://localhost:8090/v1/fluree/merge \
  -H "Content-Type: application/json" \
  -d '{"ledger": "mydb", "source": "dev", "target": "main"}'

# Non-fast-forward merge with source-winning conflict resolution
curl -X POST http://localhost:8090/v1/fluree/merge \
  -H "Content-Type: application/json" \
  -d '{"ledger": "mydb", "source": "dev", "target": "main", "strategy": "take-source"}'
```

## Merger Preview
Read-only preview of merging a source branch into a target branch. Returns the rich diff — ahead/behind commit summaries, conflict keys, and fast-forward eligibility — without mutating any nameservice or content store state.

```json
GET /merge-preview/{ledger-name}
GET /merge-preview/{ledger-name}?source={source}&target={target}&max_commits={n}&max_conflict_keys={n}&include_conflicts={bool}&include_conflict_details={bool}&strategy={strategy}
```